# 06_03 · Modelo Final — Capa 3 (AI4I, clasificación de fallo)

Construcción del **modelo de producción del proyecto**:
`StackingClassifier(LGBM afinado + RF afinado) → LogisticRegression`,
con umbral optimizado para Recall ≥ 0.85.

- Los hiperparámetros vienen de `src/utils.py` (`LGBM_BEST_PARAMS`, `RF_BEST_PARAMS`)
  — la misma fuente de verdad que usan los notebooks 05 y el script `src/training.py`,
  que es el **gemelo de este notebook** para CI/despliegue.
- Salidas: `models/final_model.pkl` (autocontenido) y `models/model_config.yaml`.

> **Prerequisito:** la cadena `05_01_*` ejecutada — en particular
> `05_01_6_Dos_Etapas` (guarda en `models.pkl` el scaler que este bundle empaqueta).


In [1]:
import sys; sys.path.insert(0, '../../src')
import warnings; warnings.filterwarnings('ignore')
import os, pickle
import numpy as np
import yaml

from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (classification_report, f1_score, precision_score,
                              recall_score, roc_auc_score)

from paths import PROCESSED, MODELS
from utils import LGBM_BEST_PARAMS, RF_BEST_PARAMS, FEATURE_COLS, eval_threshold

with open(os.path.join(PROCESSED, 'splits.pkl'), 'rb') as f:
    splits = pickle.load(f)
X_train, X_test, y_train, y_test = splits['ai4i']

with open(os.path.join(PROCESSED, 'models.pkl'), 'rb') as f:
    saved_models = pickle.load(f)
scaler = saved_models.get('stage2_scaler') or saved_models.get('scaler')
if scaler is None:
    raise KeyError(
        'models.pkl no contiene el scaler (stage2_scaler). '
        'Ejecuta antes 05_MODELOS_AFINADOS/05_01_6_Dos_Etapas_AI4I.ipynb — '
        'los notebooks deben ejecutarse en orden.'
    )

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Fallos: {y_train.sum()} train / {y_test.sum()} test')


Train: (8000, 10) | Test: (2000, 10)
Fallos: 271 train / 68 test


## 1. Entrenar el stacking de producción


In [2]:
stack = StackingClassifier(
    estimators=[
        ('lgbm', LGBMClassifier(**LGBM_BEST_PARAMS)),
        ('rf',   RandomForestClassifier(**RF_BEST_PARAMS)),
    ],
    final_estimator=LogisticRegression(class_weight='balanced',
                                       max_iter=1000, random_state=42),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    passthrough=False, n_jobs=-1,
)
print('Entrenando StackingClassifier (LGBM + RF → LogReg)...')
stack.fit(X_train, y_train)
print('Listo.')

y_prob = stack.predict_proba(X_test)[:, 1]
res = eval_threshold(y_test, y_prob)            # umbral: max F1 con Recall >= 0.85
threshold = res['Umbral']
y_pred = (y_prob >= threshold).astype(int)

metrics = {
    'precision': res['Precision'], 'recall': res['Recall'],
    'f1': res['F1'], 'roc_auc': res['ROC-AUC'],
    'fn': int(((y_pred == 0) & (y_test == 1)).sum()),
    'fp': int(((y_pred == 1) & (y_test == 0)).sum()),
}
print(f'\nUmbral de producción: {threshold:.4f}')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fallo']))
print(f'FN: {metrics["fn"]} | FP: {metrics["fp"]} | AUC: {metrics["roc_auc"]}')


Entrenando StackingClassifier (LGBM + RF → LogReg)...
Listo.

Umbral de producción: 0.9858
              precision    recall  f1-score   support

      Normal       0.99      1.00      1.00      1932
       Fallo       0.97      0.85      0.91        68

    accuracy                           0.99      2000
   macro avg       0.98      0.93      0.95      2000
weighted avg       0.99      0.99      0.99      2000

FN: 10 | FP: 2 | AUC: 0.9783


## 2. Guardar `final_model.pkl` + `model_config.yaml`


In [3]:
bundle = {
    'model':         stack,
    'threshold':     threshold,
    'scaler':        scaler,
    'feature_names': FEATURE_COLS,
    'description':   'StackingClassifier(LGBMClassifier + RandomForestClassifier)',
    'metrics':       metrics,
}
with open(os.path.join(MODELS, 'final_model.pkl'), 'wb') as f:
    pickle.dump(bundle, f)
print('Guardado: models/final_model.pkl')

config = {
    'model': {'name': 'StackingClassifier', 'type': 'ensemble',
              'base_learners': ['LGBMClassifier', 'RandomForestClassifier'],
              'meta_model': 'LogisticRegression', 'cv_folds': 5},
    'threshold': float(threshold),
    'features': FEATURE_COLS,
    'n_features': len(FEATURE_COLS),
    'hyperparameters': {'lgbm': LGBM_BEST_PARAMS, 'rf': RF_BEST_PARAMS},
    'training': {
        'dataset': 'AI4I 2020 Predictive Maintenance (UCI)',
        'n_train': int(X_train.shape[0]), 'n_test': int(X_test.shape[0]),
        'class_imbalance': '3.4% failures',
        'split_strategy': 'StratifiedShuffleSplit(test_size=0.2, random_state=42)',
        'scaling': 'StandardScaler',
    },
    'performance': {k: (float(v) if isinstance(v, float) else int(v))
                    for k, v in metrics.items()},
    'target': 'Machine failure (1=fallo, 0=normal)',
    'optimization_metric': 'Recall >= 0.85, maximize F1',
}
with open(os.path.join(MODELS, 'model_config.yaml'), 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print('Guardado: models/model_config.yaml')
print(f'\nF1={metrics["f1"]} · Recall={metrics["recall"]} · AUC={metrics["roc_auc"]}')


Guardado: models/final_model.pkl
Guardado: models/model_config.yaml

F1=0.9062 · Recall=0.8529 · AUC=0.9783
